<a href="https://colab.research.google.com/github/rxnu/LLM-Project/blob/main/3_pre_trained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline


# instantiate a sentiment-analysis pipeline with a specific model

pipe = pipeline(
    task='sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')



config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# quick inference
texts = test_df['text'].tolist()[:8]
print(texts)

['I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as they hav

In [ ]:
# run the pipeline
preds = pipe(texts)

In [ ]:
# inspect the raw output
for txt, p in zip(texts, preds):
    print(f"Review excerpt: {txt[:60]!r}…")
    print(f" → label: {p['label']}, score: {p['score']:.4f}\n")

Review excerpt: 'I love sci-fi and am willing to put up with a lot. Sci-fi mo'…
 → label: NEGATIVE, score: 0.9996

Review excerpt: 'Worth the entertainment value of a rental, especially if you'…
 → label: NEGATIVE, score: 0.6171

Review excerpt: 'its a totally average film with a few semi-alright action se'…
 → label: NEGATIVE, score: 0.9997

Review excerpt: 'STAR RATING: ***** Saturday Night **** Friday Night *** Frid'…
 → label: NEGATIVE, score: 0.9958

Review excerpt: "First off let me say, If you haven't enjoyed a Van Damme mov"…
 → label: POSITIVE, score: 0.9963

Review excerpt: 'I had high hopes for this one until they changed the name to'…
 → label: NEGATIVE, score: 0.9967

Review excerpt: 'Isaac Florentine has made some of the best western Martial A'…
 → label: NEGATIVE, score: 0.9584

Review excerpt: 'It actually pains me to say it, but this movie was horrible '…
 → label: NEGATIVE, score: 0.9994



In [ ]:
!pip install --quiet transformers datasets evaluate accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 22.7 MB/s eta 0:00:00


Fine tune Transformer


In [ ]:
!pip install -q transformers datasets evaluate

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import torch

In [ ]:
ds = load_dataset("imdb")
ds = DatasetDict({
    "train": ds["train"].shuffle(seed=42).select(range(22500)),
    "validation": ds["train"].shuffle(seed=42).select(range(22500, 25000)),
    "test": ds["test"],
})

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

ds = ds.map(tokenize_batch, batched=True)
# drop unneeded columns (raw text)
to_keep = ["input_ids", "attention_mask", "label"]
for split in ["train", "validation", "test"]:
    ds[split] = ds[split].remove_columns(
        [c for c in ds[split].column_names if c not in to_keep]
    )

# rename "label" → "labels" for Trainer compatibility
ds = ds.rename_column("label", "labels")

# format only three columns as torch tensors
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
accuracy = evaluate.load("accuracy")
f1       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       f1.compute(predictions=preds, references=labels)["f1"],
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="imdb-distilbert-run1",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    report_to="none",
    run_name=None,
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.280400,0.224887,0.908800,0.911559
2,0.172900,0.285905,0.910800,0.911261
3,0.099300,0.327602,0.916800,0.919815


TrainOutput(global_step=4221, training_loss=0.1926606039785147, metrics={'train_runtime': 1539.4433, 'train_samples_per_second': 43.847, 'train_steps_per_second': 2.742, 'total_flos': 4470774704640000.0, 'train_loss': 0.1926606039785147, 'epoch': 3.0})

In [ ]:
metrics = trainer.evaluate(ds["validation"])
print("Validation metrics:", metrics)

Validation metrics: {'eval_loss': 0.3276020586490631, 'eval_accuracy': 0.9168, 'eval_f1': 0.9198149575944488, 'eval_runtime': 16.8197, 'eval_samples_per_second': 148.635, 'eval_steps_per_second': 4.697, 'epoch': 3.0}


In [ ]:
trainer.save_model("imdb-distilbert-finetuned")
tokenizer.save_pretrained("imdb-distilbert-finetuned")


('imdb-distilbert-finetuned/tokenizer_config.json',
 'imdb-distilbert-finetuned/special_tokens_map.json',
 'imdb-distilbert-finetuned/vocab.txt',
 'imdb-distilbert-finetuned/added_tokens.json',
 'imdb-distilbert-finetuned/tokenizer.json')

In [ ]:
# evaluate on the official test split
test_metrics = trainer.evaluate(ds["test"])
print("Test set metrics:", test_metrics)

Test set metrics: {'eval_loss': 0.3494538962841034, 'eval_accuracy': 0.91364, 'eval_f1': 0.9139600685450125, 'eval_runtime': 171.7369, 'eval_samples_per_second': 145.572, 'eval_steps_per_second': 4.553, 'epoch': 3.0}


In [ ]:
#load to HF
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
 !apt-get install git-lfs

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer


model     = AutoModelForSequenceClassification.from_pretrained(
    "./imdb-distilbert-finetuned", local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "./imdb-distilbert-finetuned", local_files_only=True
)

model.push_to_hub(repo_id="rxnu/imdb-distilbert-finetuned")
tokenizer.push_to_hub(repo_id="rxnu/imdb-distilbert-finetuned")




Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-finetuned/commit/33f693c4c780a1970c96739239874e1eeed34612', commit_message='Upload tokenizer', commit_description='', oid='33f693c4c780a1970c96739239874e1eeed34612', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-finetuned'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
trainer.save_model("/content/drive/MyDrive/imdb-distilbert-finetuned")
tokenizer.save_pretrained("/content/drive/MyDrive/imdb-distilbert-finetuned")


('/content/drive/MyDrive/imdb-distilbert-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/vocab.txt',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/added_tokens.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/tokenizer.json')